# Explainable AI for Maternal Health Risk Stratification in Kenya
### Predicting Missed Timely Postnatal Care (PNC) — CRISP-DM Notebook

This notebook follows the CRISP-DM methodology end-to-end. Each phase is its own top-level section (`##`) so the whole team works in one file instead of separate notebooks.

**Phases:** 1. Business Understanding · 2. Data Understanding · 3. Data Preparation · 4. Modeling · 5. Evaluation · 6. Deployment

## **Explainable Machine Learning for Predicting Missed Timely Postnatal Care in Kenya**

## 1. Business Understanding

## 1.1 Business Overview

Maternal healthcare is a continuum that includes antenatal care (ANC), delivery care, and postnatal care (PNC). While Kenya has made considerable progress in maternal healthcare utilization, gaps remain in the continuity of care after childbirth.

According to the Kenya Demographic and Health Survey (KDHS) 2022, approximately 98% of women who had a live birth or stillbirth in the two years preceding the survey received ANC from a skilled provider, while about two-thirds attended at least four ANC visits. In addition, 88% of live births occurred in a health facility. However, only 78% of women with a live birth in the two years preceding the survey received a postnatal check within the first two days after birth.

These figures indicate that although contact with the health system is relatively high during pregnancy and delivery, timely postnatal follow-up is not universal. The gap is also uneven across population groups. For example, the KDHS 2022 reports that timely postnatal checks were received by 79% of women in urban areas compared with 69% in rural areas, while coverage ranged from 59% among women in the lowest wealth quintile to 83% among those in the highest wealth quintile.

The availability of detailed maternal-health data from the KDHS 2022 creates an opportunity to use data science to identify characteristics associated with missed timely postnatal care and support more targeted follow-up.

## 1.2  Problem Statement

Despite high utilization of antenatal and facility-based delivery services in Kenya, a proportion of women do not receive a postnatal health check within the first two days after childbirth. Current aggregate statistics describe the size of the gap but do not identify which individual women are more likely to miss timely postnatal care.

Healthcare workers responsible for community-level follow-up may therefore have limited information to prioritize women who could benefit most from additional follow-up.

This project seeks to address this gap by developing an explainable machine-learning model that predicts the likelihood of a woman missing timely postnatal care using demographic, socioeconomic, ANC, delivery, and healthcare-access characteristics from the KDHS 2022 dataset.

The model will not replace healthcare professionals or provide a clinical diagnosis. Instead, it will serve as a decision-support prototype that can help maternal and child health teams identify higher-risk profiles and understand the factors contributing to each prediction.

## 1.3 Business Objective

**Main Objective**

- To develop an explainable binary classification model that predicts the likelihood of women missing timely postnatal care in Kenya and provides understandable explanations of the factors contributing to each prediction.

**Specific Objectives**

- To identify and prepare relevant demographic, socioeconomic, ANC, delivery, and healthcare-access variables from the KDHS 2022 dataset.
- To examine the relationship between these factors and timely postnatal-care utilization.
- To develop a Logistic Regression model as a baseline for predicting missed timely postnatal care.
- To develop an XGBoost classification model and compare its predictive performance with the baseline model.
- To evaluate model performance using PR-AUC, recall, precision, F1-score, and ROC-AUC.
- To apply SHAP to explain the factors influencing individual predictions and overall model behaviour.
- To develop an interactive prototype that provides a predicted risk score and the main factors contributing to the prediction.
- To visualize predicted risk at DHS cluster level using the available GPS data as a presentation layer.

## 1.4 Stakeholders

**Stakeholder	Role / Interest**

- County Ministry of Health MCH Coordinators - Monitor maternal-health outcomes and coordinate targeted interventions

- Community Health Promoters (CHPs) - Conduct community and household follow-up and potentially prioritize women for visits

- Ministry of Health - Use evidence to support maternal-health planning and resource allocation

- Healthcare facilities - Strengthen continuity between ANC, delivery and PNC

- NGOs and development partners - Target maternal-health programmes and resources

- Researchers and public-health analysts - Use evidence to understand factors associated with PNC utilization

- Women and newborns - Ultimate beneficiaries of improved continuity of maternal and newborn care

## 1.5 Metrics of Success

The success of the project will be evaluated at both technical and practical levels.

**Model Performance**

- The primary metric will be PR-AUC, because the project focuses on identifying the minority group of women who miss timely postnatal care.

**Additional metrics will include:**

- Recall — ability to identify women who miss timely PNC
- Precision — proportion of women identified as high risk who actually miss timely PNC
- F1-score — balance between precision and recall
- ROC-AUC — overall discrimination
- Confusion matrix — assessment of classification errors

**Explainability**

- The model should be able to provide meaningful explanations for individual predictions using SHAP, showing the major factors that increase or decrease predicted risk.

**Practical Success**

The final prototype will be considered successful if a user can:

Enter a woman's profile → receive a predicted probability of missing timely PNC → see the risk classification → understand the main factors contributing to the prediction.

## 2. Data Understanding

In [ ]:
# Portable setup: read the repository ZIP files directly (no extraction required)
from pathlib import Path
import io
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in [cwd, *cwd.parents] if (folder / "data" / "raw" / "KENR8CDT.zip").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/raw/KENR8CDT.zip. Run this notebook from inside the cloned repository."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MATERNAL_ZIP = RAW_DIR / "KENR8CDT.zip"
GPS_ZIP = RAW_DIR / "KEGE8AFL.zip"

for archive_path in (MATERNAL_ZIP, GPS_ZIP):
    if not archive_path.exists():
        raise FileNotFoundError(f"Missing required archive: {archive_path.relative_to(PROJECT_ROOT)}")

print(f"Maternal source: {MATERNAL_ZIP.relative_to(PROJECT_ROOT)}")
print(f"GPS source:      {GPS_ZIP.relative_to(PROJECT_ROOT)}")

In [ ]:
# Load the pregnancy and postnatal-care recode directly from its ZIP archive.
with zipfile.ZipFile(MATERNAL_ZIP) as archive:
    stata_name = next(name for name in archive.namelist() if name.upper().endswith(".DTA"))
    stata_bytes = io.BytesIO(archive.read(stata_name))

reader = pd.io.stata.StataReader(stata_bytes)
df = reader.read(convert_categoricals=False)
variable_labels = reader.variable_labels()
value_labels = reader.value_labels()

print(f"Loaded: {df.shape[0]:,} records x {df.shape[1]} variables")

In [ ]:
# Verify the GPS archive without loading coordinates as model inputs.
with zipfile.ZipFile(GPS_ZIP) as archive:
    gps_shapefile = next(name for name in archive.namelist() if name.upper().endswith(".SHP"))

print(f"Verified GPS file inside ZIP: {gps_shapefile}")
print("GPS coordinates are reserved for mapping and are not model predictors.")

## Eligible Population

This recode is **pregnancy-indexed**, not birth-indexed. The eligibility variable is `m80`
("pregnancy outcome for this section"), which flags each woman's most recent pregnancy outcome:

| Code | Meaning |
|---|---|
| 1 | Most recent live birth |
| 3 | Most recent stillbirth |

Filtering to `m80 ∈ {1, 3}` restricts the data to each woman's most recent live birth or stillbirth —
the population this project defines as eligible for postnatal-care risk stratification. Stillbirths
are retained by design: mothers who experience a stillbirth still require postpartum care.

In [4]:
# Full breakdown of ALL pregnancy outcomes in the raw data, before any filtering.
# This shows exactly what gets excluded and why: prior pregnancies and miscarriages/abortions
# are out of scope, since DHS only collects postnatal-care data for a woman's most recent
# pregnancy outcome.
m80_labels = {
    1: "Most recent live birth",
    2: "Prior live birth",
    3: "Most recent stillbirth",
    4: "Prior stillbirth",
    5: "Miscarriage/abortion"
}

outcome_counts = df["m80"].value_counts().sort_index()
outcome_summary = pd.DataFrame({
    "Code": outcome_counts.index,
    "Meaning": [m80_labels.get(c, "Unknown") for c in outcome_counts.index],
    "Count": outcome_counts.values
})
outcome_summary["% of total"] = (outcome_summary["Count"] / len(df) * 100).round(1)

print(f"Total records before filtering: {len(df):,}\n")
print(outcome_summary.to_string(index=False))

Total records before filtering: 13,184

 Code                Meaning  Count  % of total
    1 Most recent live birth  10412        79.0
    2       Prior live birth   1317        10.0
    3 Most recent stillbirth    194         1.5
    4       Prior stillbirth     13         0.1
    5   Miscarriage/abortion   1248         9.5


Applying the `m80 ∈ {1, 3}` filter gives the eligible population, along with a check for any women who appear more than once (multiple "most recent" outcomes recorded across different interview sections).

In [5]:
# Restrict to each woman's most recent live birth (m80=1) or stillbirth (m80=3).
# Stillbirths are retained by design (see markdown above) - mothers still need postpartum care.
eligible = df[df["m80"].isin([1, 3])].copy()

# Breakdown of the eligible population after filtering
print(f"Eligible records (most recent live birth or stillbirth): {eligible.shape[0]:,}")
print(f"Unique women: {eligible['caseid'].nunique():,}")
print(f"Unique survey clusters: {eligible['v001'].nunique():,}")
print()

# Confirm the split between live births and stillbirths within the eligible set
eligible_breakdown = eligible["m80"].value_counts().sort_index()
eligible_summary = pd.DataFrame({
    "Code": eligible_breakdown.index,
    "Meaning": [m80_labels.get(c, "Unknown") for c in eligible_breakdown.index],
    "Count": eligible_breakdown.values
})
eligible_summary["% of eligible"] = (eligible_summary["Count"] / len(eligible) * 100).round(1)

print(eligible_summary.to_string(index=False))
print()

# Note: a woman can appear more than once here if DHS recorded two separate "most recent"
# outcomes for her (e.g. a recent live birth AND a separate recent pregnancy-loss entry).
# We flag this now and resolve it after the target is built (see below), since we need the
# target value to decide which of a woman's records to keep if we ever need to pick one.
duplicate_women = eligible["caseid"].value_counts()
duplicate_women = duplicate_women[duplicate_women > 1]
print(f"Women appearing more than once in the eligible set: {len(duplicate_women)}")

Eligible records (most recent live birth or stillbirth): 10,606
Unique women: 10,540
Unique survey clusters: 1,680

 Code                Meaning  Count  % of eligible
    1 Most recent live birth  10412           98.2
    3 Most recent stillbirth    194            1.8

Women appearing more than once in the eligible set: 66


## Target Variable: Missed Timely Postnatal Care

The mother's postnatal check is captured across two DHS routing paths:
- **Pre-discharge check** (facility path): `m62` (checked, yes/no) → `m63` (timing)
- **Post-discharge / home-delivery check** (catch-all path): `m66` (checked, yes/no) → `m67` (timing)

Timing codes follow the DHS convention: `1XX` = hours, `2XX` = days, `3XX` = weeks, `998` = don't know.

**Target rule** (any provider — matches the standard global PNC-timeliness indicator, which does not
restrict by provider type):
- **0 (timely):** a check occurred within 48 hours on either path (hours `100`–`148`, or days `201`–`202`)
- **1 (missed):** no check occurred on either path, or the earliest check occurred at 3+ days / weeks
- **Excluded:** timing coded `998` (don't know) — only 40 records (0.38%), too small to justify an
  imputation assumption

In [6]:
# Classify each routing path (pre-discharge and post-discharge/home) as timely, late,
# no check occurred, or not applicable - then combine into the final binary target.
def classify_check(check_flag, timing_code):
    if check_flag == 1:
        if pd.isna(timing_code) or timing_code == 998:
            return "unknown"
        elif 100 <= timing_code <= 148 or 201 <= timing_code <= 202:
            return "timely"
        elif timing_code >= 203:
            return "late"
        return "unknown"
    elif check_flag == 0:
        return "no_check"
    return "not_applicable"

def combine_target(pre, post):
    if pre == "timely" or post == "timely":
        return 0
    if pre == "unknown" or post == "unknown":
        return np.nan
    return 1

eligible["pre_discharge_status"] = eligible.apply(lambda r: classify_check(r["m62"], r["m63"]), axis=1)
eligible["post_discharge_status"] = eligible.apply(lambda r: classify_check(r["m66"], r["m67"]), axis=1)
eligible["missed_timely_pnc"] = eligible.apply(
    lambda r: combine_target(r["pre_discharge_status"], r["post_discharge_status"]), axis=1
)

# Exclude the small number of records where timing could not be determined (998/DK)
model_df = eligible[eligible["missed_timely_pnc"].notna()].copy()
model_df["missed_timely_pnc"] = model_df["missed_timely_pnc"].astype(int)

print(f"Final modelling population (before dedup): {model_df.shape[0]:,} records")
print(model_df["missed_timely_pnc"].value_counts(normalize=True).mul(100).round(1))

Final modelling population (before dedup): 10,570 records
missed_timely_pnc
0    74.3
1    25.7
Name: proportion, dtype: float64


### Reconciling the Record Count and Checking for Duplicate Women

Two things to confirm before this becomes the final modelling dataset:
1. Exactly how many records were excluded when building the target, and why.
2. Whether any woman appears more than once (66 were flagged earlier, in the full eligible
   set) - and if so, keep only her most recent pregnancy record, so each woman contributes
   exactly one observation to the model.

In [7]:
# Confirm exactly why the record count drops from 10,606 (eligible) to a smaller number here:
# every record where m63/m67 timing was coded 998 (Don't know) was excluded from the target,
# since we can't safely assign timely/missed without guessing.
n_eligible = eligible.shape[0]
n_unknown = eligible["missed_timely_pnc"].isna().sum()
n_model = model_df.shape[0]

print(f"Eligible population:           {n_eligible:,}")
print(f"Excluded - unknown target (m63/m67 = 998, Don't know): {n_unknown}")
print(f"Modelling population (before dedup): {n_model:,}")
assert n_eligible - n_unknown == n_model, "Numbers don't reconcile - check filtering logic"
print(f"\nConfirmed: {n_eligible:,} - {n_unknown} unknown-target rows = {n_model:,}")

Eligible population:           10,606
Excluded - unknown target (m63/m67 = 998, Don't know): 36
Modelling population (before dedup): 10,570

Confirmed: 10,606 - 36 unknown-target rows = 10,570


In [8]:
# Check for women appearing more than once in the modelling population
dup_counts = model_df["caseid"].value_counts()
dup_women = dup_counts[dup_counts > 1]

print(f"Women appearing more than once: {len(dup_women)}")
print(f"Total records involved: {dup_women.sum()}")

if len(dup_women) > 0:
    # p18 = century day code (cdc) of pregnancy outcome - a precise, sortable chronological
    # value used to determine which of a woman's records is truly the most recent.
    sample = model_df[model_df["caseid"].isin(dup_women.index)][
        ["caseid", "m80", "p1", "p2", "p18", "missed_timely_pnc"]
    ].sort_values(["caseid", "p18"])
    print("\nSample of duplicate records (chronological order via p18):")
    print(sample.head(20))
    print(f"\np18 missing for {sample['p18'].isna().sum()} of these rows")

Women appearing more than once: 65
Total records involved: 130

Sample of duplicate records (chronological order via p18):
               caseid  m80  p1    p2    p18  missed_timely_pnc
37           8  71  2    3   1  2021  44206                  0
36           8  71  2    1  12  2021  44548                  0
119         18  67  2    3   4  2021  44290                  1
118         18  67  2    1   2  2022  44598                  1
305         43  46 15    3   7  2019  43674                  0
304         43  46 15    1  10  2020  44115                  0
712         98   8  6    3   1  2020  43847                  0
711         98   8  6    1   2  2021  44248                  1
861        116  12  2    3   8  2020  44047                  1
860        116  12  2    1   8  2021  44413                  0
1107       136  12  2    3   2  2020  43870                  1
1106       136  12  2    1  11  2021  44503                  1
1155       139  56  2    3   3  2020  43906               

Note: this shows **65** duplicate women here, one fewer than the **66** found earlier in the
full eligible set. That's expected: one of the original 66 had one of her two records excluded
just above (unknown target timing), leaving her with only one record in `model_df` - so she no
longer appears as a "duplicate" at this point, even though she started as one. This is confirmed
explicitly further down.

### Deduplicating to One Record per Woman

Each woman should contribute exactly one observation to the model - both to avoid distorting
the class balance and to prevent the same woman's data leaking across train/test splits later.
For any woman with more than one eligible record, we keep only the most recent pregnancy,
using `p18` (century day code of pregnancy outcome) as the chronological key, with `p2`/`p1`
(year/month of outcome) as a fallback if `p18` is missing for a given row. `p18` is fully
populated for these rows, so no fallback is actually needed here.

In [9]:
import numpy as np

def pregnancy_date_key(row):
    if pd.notna(row.get("p18")):
        return row["p18"]
    if pd.notna(row.get("p2")) and pd.notna(row.get("p1")):
        return row["p2"] * 12 + row["p1"]  # fallback: year*12 + month
    return np.nan

model_df["_date_key"] = model_df.apply(pregnancy_date_key, axis=1)

before_n = model_df.shape[0]
before_women = model_df["caseid"].nunique()

# Keep only the most recent record per woman
model_df = (
    model_df.sort_values("_date_key", ascending=False)
    .drop_duplicates(subset="caseid", keep="first")
    .drop(columns="_date_key")
)

print(f"Before dedup: {before_n:,} records, {before_women:,} unique women")
print(f"After dedup:  {model_df.shape[0]:,} records, {model_df['caseid'].nunique():,} unique women")
print(f"Records dropped: {before_n - model_df.shape[0]}")
print()
print("Final target distribution after dedup:")
print(model_df["missed_timely_pnc"].value_counts(normalize=True).mul(100).round(1))

Before dedup: 10,570 records, 10,505 unique women
After dedup:  10,505 records, 10,505 unique women
Records dropped: 65

Final target distribution after dedup:
missed_timely_pnc
0    74.4
1    25.6
Name: proportion, dtype: float64


### Confirming the Final Unique-Women Count

After deduplication, `model_df` has one record per woman. We'd expect this to match the
10,540 unique women found in the eligible population - let's check.

In [10]:
# %% Confirm final unique-women count against the eligible population's 10,540
print(f"Final unique women in model_df: {model_df['caseid'].nunique():,}")
print(f"Expected (from eligible population): 10,540")
print(f"Difference: {10540 - model_df['caseid'].nunique()} (women who lost both records to unknown-target exclusion)")

Final unique women in model_df: 10,505
Expected (from eligible population): 10,540
Difference: 35 (women who lost both records to unknown-target exclusion)


### Explaining the Gap

The final count is short by 35 women. To confirm why, we check whether the excluded
unknown-target records (36 total) belong to women who had a second, valid record to fall back
on (originally-duplicate women), or to women whose *only* eligible record was excluded outright.

In [11]:
# Identify which unknown-target record belongs to a duplicate woman vs. a single-record woman
# IMPORTANT: use duplicates from the full eligible set (before target exclusion), not model_df,
# since a woman who lost one of her two records to unknown-target exclusion would no longer
# appear as "duplicate" in model_df even though she originally had 2 eligible records.
eligible_dup_counts = eligible["caseid"].value_counts()
eligible_dup_caseids = set(eligible_dup_counts[eligible_dup_counts > 1].index)

unknown_records = eligible[eligible["missed_timely_pnc"].isna()]
unknown_from_dup_women = unknown_records[unknown_records["caseid"].isin(eligible_dup_caseids)]
unknown_from_single_women = unknown_records[~unknown_records["caseid"].isin(eligible_dup_caseids)]

print(f"Unknown-target records from (originally) duplicate women: {len(unknown_from_dup_women)}")
print(f"Unknown-target records from single-record women: {len(unknown_from_single_women)}")

Unknown-target records from (originally) duplicate women: 1
Unknown-target records from single-record women: 35


### Final Modelling Population - Summary

Putting the full chain together:

`13,184` total records → `10,606` eligible (most recent live birth/stillbirth) → `10,570`
after excluding 36 unknown-target records → **`10,505`** after deduplication (65 duplicate
records removed, one row kept per woman).

**Final validated class balance:** 74.4% timely / 25.6% missed timely PNC, across 10,505 women
- each woman contributing exactly one observation.

A stricter "quality PNC" indicator (requiring *all* components of care, not just one qualifying
check) from independent KDHS 2022 analyses sits at 32.7–39% nationally — consistent with our
basic timeliness measure landing meaningfully higher, since it only requires a single check
within 48 hours.

## Candidate Predictors and Missingness

Predictors were mapped from the project proposal to their DHS variable codes, then checked for
missingness across the full eligible population (10,606 records) - independent of target
construction or deduplication, since predictor availability is a property of the raw data itself.

In [12]:
predictor_vars = {
    "v012": "Maternal age", "v106": "Education level", "v501": "Marital status",
    "v714": "Employment status", "v190": "Household wealth", "v201": "Children ever born",
    "m10": "Pregnancy intention", "m14": "Number of ANC visits", "m13": "Timing of first ANC visit",
    "m15": "Place of delivery", "m17": "Caesarean delivery", "v025": "Urban/rural residence",
    "v024": "Region"
}

summary = []
for var, label in predictor_vars.items():
    n_missing = eligible[var].isna().sum()
    summary.append({
        "Variable": var, "Description": label,
        "Missing": n_missing, "Missing %": round(100 * n_missing / len(eligible), 1)
    })

pd.DataFrame(summary)

,Variable,Description,Missing,Missing %
0,v012,Maternal age,0,0.0
1,v106,Education level,0,0.0
2,v501,Marital status,0,0.0
3,v714,Employment status,0,0.0
4,v190,Household wealth,0,0.0
5,v201,Children ever born,0,0.0
6,m10,Pregnancy intention,0,0.0
7,m14,Number of ANC visits,0,0.0
8,m13,Timing of first ANC visit,399,3.8
9,m15,Place of delivery,0,0.0


## Key Findings Summary

1. **Eligible population confirmed:** 10,606 pregnancy records (10,540 women, 1,680 clusters),
   matching the proposal's expected ~10,603.
2. **Target constructed and validated:** 36 records excluded (unknown timing), leaving 10,570
   with a usable target - 74.3% timely / 25.7% missed timely PNC.
3. **Deduplicated to one record per woman:** 65 duplicate records removed (of 66 originally
   identified; the 66th woman had already lost one of her two records to target exclusion),
   leaving **10,505 records - the final modelling population** - at 74.4% timely / 25.6% missed,
   consistent with the pre-dedup split.
4. **Most core predictors are fully populated** (0% missing): age, education, marital status,
   employment, wealth, parity, pregnancy intention, ANC visit count, delivery place, caesarean status,
   residence, region.
5. **Structural gap identified:** healthcare-access-barrier variables (permission/money/distance/going
   alone) were only asked to half the sample (DHS long/short questionnaire split) — excluded from the
   main predictor set, noted as a limitation.
6. **33 unused "country-specific" template columns** (100% missing) identified for removal during
   data preparation.

## 3. Data Preparation and Cleaning

This section converts the validated population from Data Understanding into the final modelling table. It keeps exactly **13 approved predictors and one binary target**. Identifiers, survey-design fields, auditing fields, duplicate geographic representations, and target-source variables are excluded from the exported model inputs.

### 3.1 Start from the validated one-woman table

The preceding section has already applied the eligibility rule, constructed the target, excluded uncertain outcomes, and retained one usable record per woman. We copy that table so cleaning operations do not alter the earlier audit trail.

In [ ]:
from pathlib import Path

clean = model_df.copy()

# Locate the repository root so the export works from the root or notebooks folder.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in [cwd, *cwd.parents] if (folder / "data" / "raw" / "KENR8CDT.zip").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/raw/KENR8CDT.zip. Run this notebook from inside the cloned repository."
    )

print(f"Validated rows entering preparation: {len(clean):,}")
print(f"Unique women: {clean['caseid'].nunique():,}")

### 3.2 Clean special codes and create understandable fields

DHS uses numeric codes to store categories. The next cell:

- changes `98 = Don't know` to a true missing value for ANC timing and visits;
- groups detailed delivery places into useful categories;
- adds readable labels for the main categorical predictors.

Missing predictor values are **kept** for now. Imputation should be learned from training data only, after the train/test split, to avoid data leakage.

In [ ]:
# Replace DHS special codes with true missing values.
clean["m13"] = clean["m13"].replace({98: np.nan, 99: np.nan})
clean["m14"] = clean["m14"].replace({98: np.nan, 99: np.nan})

education = {0: "No education", 1: "Primary", 2: "Secondary", 3: "Higher"}
marital = {
    0: "Never married", 1: "Married", 2: "Living with partner",
    3: "Widowed", 4: "Divorced", 5: "Separated",
}
wealth = {1: "Poorest", 2: "Poorer", 3: "Middle", 4: "Richer", 5: "Richest"}
yes_no = {0: "No", 1: "Yes"}
residence = {1: "Urban", 2: "Rural"}
intention = {1: "Wanted then", 2: "Wanted later", 3: "Wanted no more"}
county = {
    1: "Mombasa", 2: "Kwale", 3: "Kilifi", 4: "Tana River", 5: "Lamu",
    6: "Taita Taveta", 7: "Garissa", 8: "Wajir", 9: "Mandera", 10: "Marsabit",
    11: "Isiolo", 12: "Meru", 13: "Tharaka-Nithi", 14: "Embu", 15: "Kitui",
    16: "Machakos", 17: "Makueni", 18: "Nyandarua", 19: "Nyeri", 20: "Kirinyaga",
    21: "Murang'a", 22: "Kiambu", 23: "Turkana", 24: "West Pokot", 25: "Samburu",
    26: "Trans Nzoia", 27: "Uasin Gishu", 28: "Elgeyo-Marakwet", 29: "Nandi",
    30: "Baringo", 31: "Laikipia", 32: "Nakuru", 33: "Narok", 34: "Kajiado",
    35: "Kericho", 36: "Bomet", 37: "Kakamega", 38: "Vihiga", 39: "Bungoma",
    40: "Busia", 41: "Siaya", 42: "Kisumu", 43: "Homa Bay", 44: "Migori",
    45: "Kisii", 46: "Nyamira", 47: "Nairobi",
}


def delivery_place_group(code):
    if pd.isna(code):
        return pd.NA
    code = int(code)
    if 10 <= code <= 19:
        return "Home"
    if 20 <= code <= 29:
        return "Public facility"
    if 30 <= code <= 39:
        return "Private facility"
    if 40 <= code <= 49:
        return "Faith-based/NGO facility"
    if code == 96:
        return "Other"
    return pd.NA


model_data = pd.DataFrame({
    "maternal_age": clean["v012"].astype("Int64"),
    "county_name": clean["v024"].map(county),
    "residence": clean["v025"].map(residence),
    "education_level": clean["v106"].map(education),
    "wealth_quintile": clean["v190"].map(wealth),
    "children_ever_born": clean["v201"].astype("Int64"),
    "marital_status": clean["v501"].map(marital),
    "currently_working": clean["v714"].map(yes_no),
    "pregnancy_intention": clean["m10"].map(intention),
    "first_anc_month": clean["m13"].astype("Int64"),
    "anc_visits": clean["m14"].astype("Int64"),
    "delivery_place": clean["m15"].map(delivery_place_group),
    "caesarean_delivery": clean["m17"].map(yes_no),
    "missed_timely_pnc": clean["missed_timely_pnc"].astype("int8"),
})

display(model_data.head())

### 3.3 Create a separate visualization table

The modelling table stays limited to 13 predictors and one target. For charts, maps, and survey-representative summaries, we create a separate table that restores only four useful context fields:

- `cluster_id` for later joins to the displaced DHS mapping file;
- `sample_weight` for representative percentages and rates;
- `county_code` for ordered geographic joins;
- `pregnancy_outcome` for comparing live births and stillbirths.

Woman, household, and respondent identifiers are deliberately excluded because they are unnecessary for visualizations.

In [ ]:
visualization_context = pd.DataFrame({
    "cluster_id": clean["v001"].astype("Int64"),
    "sample_weight": clean["v005"] / 1_000_000,
    "county_code": clean["v024"].astype("Int64"),
    "pregnancy_outcome": clean["m80"].map({1: "Live birth", 3: "Stillbirth"}),
})

visualization_data = pd.concat([visualization_context, model_data], axis=1)

assert visualization_data.shape == (len(model_data), 18)
assert visualization_data["sample_weight"].gt(0).all()
assert visualization_data["cluster_id"].notna().all()
assert visualization_data["county_code"].between(1, 47).all()

print(f"Model table:         {model_data.shape[0]:,} rows x {model_data.shape[1]} columns")
print(f"Visualization table: {visualization_data.shape[0]:,} rows x {visualization_data.shape[1]} columns")
display(visualization_data.head())

### 3.4 Validate the cleaned table

These checks act like guardrails. If a future file behaves differently, the notebook stops instead of silently producing a questionable dataset.

In [ ]:
assert len(model_data) == clean["caseid"].nunique(), "The final table is not one row per woman."
assert model_data.shape[1] == 14, "Expected 13 predictors and one target."
assert model_data["missed_timely_pnc"].isin([0, 1]).all(), "Target contains values other than 0/1."
assert model_data["maternal_age"].between(15, 49).all(), "Maternal age is outside 15–49."
assert model_data["county_name"].notna().all(), "A county code could not be translated."
assert model_data["first_anc_month"].dropna().between(1, 10).all(), "ANC month is implausible."
assert model_data["anc_visits"].dropna().between(0, 20).all(), "ANC visit count is implausible."

target_source_fields = {"m62", "m63", "m66", "m67", "pre_discharge_status", "post_discharge_status"}
assert target_source_fields.isdisjoint(model_data.columns), "Target-source fields leaked into the model table."

non_model_fields = {
    "woman_id", "cluster_id", "household_id", "respondent_line", "sample_weight",
    "county_code", "pregnancy_outcome",
}
assert non_model_fields.isdisjoint(model_data.columns), "A non-model field remains in the export."

validation = pd.DataFrame({
    "check": [
        "Rows in final table", "Unique women", "Columns in final table", "Predictor columns",
        "Missing target values", "Target values", "Target leakage fields", "Non-model fields",
    ],
    "result": [
        f"{len(model_data):,}", f"{clean['caseid'].nunique():,}", model_data.shape[1], model_data.shape[1] - 1,
        model_data["missed_timely_pnc"].isna().sum(),
        sorted(model_data["missed_timely_pnc"].unique().tolist()),
        sorted(target_source_fields.intersection(model_data.columns)),
        sorted(non_model_fields.intersection(model_data.columns)),
    ],
})
display(validation)
print("All validation checks passed.")

### 3.5 Review missing information

This table shows which fields still have blanks after special-code cleaning. Blanks are not automatically errors: for example, ANC timing can be missing when no ANC visit occurred.

We deliberately avoid filling these values here. During modelling, numeric and categorical imputation should be fitted using the training set only.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_rows": model_data.isna().sum(),
    "missing_percent": (100 * model_data.isna().mean()).round(2),
}).sort_values(["missing_rows", "missing_percent"], ascending=False)

display(missing_summary.loc[missing_summary["missing_rows"] > 0])

### 3.6 Save the model-ready dataset

The final CSV contains only the **13 approved model predictors** and the binary target. Identifiers, survey-design fields, duplicate geographic representations, pregnancy-outcome auditing fields, and raw postnatal routing fields are excluded.

In [ ]:
OUTPUT_RELATIVE_PATH = Path("data") / "processed" / "maternal_pnc_model_inputs.csv"
OUTPUT_FILE = PROJECT_ROOT / OUTPUT_RELATIVE_PATH
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

try:
    model_data.to_csv(OUTPUT_FILE, index=False)
except PermissionError:
    from datetime import datetime

    fallback_name = f"maternal_pnc_model_inputs_{datetime.now():%Y%m%d_%H%M%S}.csv"
    OUTPUT_RELATIVE_PATH = Path("data") / "processed" / fallback_name
    OUTPUT_FILE = PROJECT_ROOT / OUTPUT_RELATIVE_PATH
    model_data.to_csv(OUTPUT_FILE, index=False)
    print("The standard output file is open in another program, so a timestamped copy was saved instead.")

print(f"Saved {len(model_data):,} rows and {model_data.shape[1]} columns to:")
print(OUTPUT_RELATIVE_PATH.as_posix())

### 3.7 Save the visualization dataset

This separate export supports weighted charts and geographic summaries without accidentally expanding the feature matrix used by the model.

In [ ]:
VISUALIZATION_RELATIVE_PATH = Path("data") / "processed" / "maternal_pnc_visualization_data.csv"
VISUALIZATION_FILE = PROJECT_ROOT / VISUALIZATION_RELATIVE_PATH

try:
    visualization_data.to_csv(VISUALIZATION_FILE, index=False)
except PermissionError:
    from datetime import datetime

    fallback_name = f"maternal_pnc_visualization_data_{datetime.now():%Y%m%d_%H%M%S}.csv"
    VISUALIZATION_RELATIVE_PATH = Path("data") / "processed" / fallback_name
    VISUALIZATION_FILE = PROJECT_ROOT / VISUALIZATION_RELATIVE_PATH
    visualization_data.to_csv(VISUALIZATION_FILE, index=False)
    print("The standard visualization file is open, so a timestamped copy was saved instead.")

print(f"Saved {len(visualization_data):,} rows and {visualization_data.shape[1]} columns to:")
print(VISUALIZATION_RELATIVE_PATH.as_posix())

### 3.8 Cleaning summary

The cleaning flow is now fully reproducible:

1. Load only required fields from the original DHS archive.
2. Keep the most recent live birth or stillbirth records.
3. derive the missed-timely-PNC outcome from both DHS care routes.
4. Exclude only outcomes whose timing cannot be known safely.
5. Keep one usable record per woman.
6. Convert DHS special codes and add readable labels.
7. Validate and export a model-ready table.

The next modelling notebook should split this data into training and test sets **before** learning any imputation, encoding, or scaling rules.

## 4. Modeling

*(XGBoost baseline + tuning.)*

In [18]:
# TODO: train/test split + XGBoost model

## 5. Evaluation

*(Metrics + SHAP explainability.)*

In [19]:
# TODO: evaluation metrics + SHAP plots

## 6. Deployment

*(Streamlit app — see `/app`.)*

In [20]:
# TODO: link/summary of Streamlit deployment